# Mcardiac_EA training

Edit data, checkpoint, and output paths in `config.yaml` beside this notebook. Relative paths start from the experiment directory. Review routes, `steps`, and training options in the setup and training cells below, then run from top to bottom.


In [ ]:
from pathlib import Path
import yaml

DATASET = 'heart_slide_seq'
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / "Mcardiac_EA"
REPO_ROOT = EXPERIMENT_DIR.parents[1]
import sys
sys.path.append(str(REPO_ROOT / "src"))

CONFIG = yaml.safe_load(
    (EXPERIMENT_DIR / "config.yaml").read_text(encoding="utf-8")
)

def experiment_path(value):
    path = Path(value)
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

DATA_ROOT = experiment_path(CONFIG["data_root"])
RUN_ROOT = experiment_path(CONFIG["run_root"])
CHECKPOINT_ROOT = experiment_path(CONFIG["checkpoint_root"])
SCANVI_DIR = experiment_path(CONFIG["scanvi_dir"])
SCANVI_ADATA = experiment_path(CONFIG["input_path"])
STAGE1_CHECKPOINT_ROOT = experiment_path(CONFIG["stage1_checkpoint"])
STAGE2_CHECKPOINT_ROOT = experiment_path(CONFIG["stage2_checkpoint"])
LR_PAIRS = experiment_path(CONFIG["lr_pairs_path"]) if CONFIG["lr_pairs_path"] else None
DIFF_MAP = experiment_path(CONFIG["diff_map_path"]) if "diff_map_path" in CONFIG else None
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import re
import scanpy as sc
import torch
import numpy as np
import pandas as pd
import sys

import scvi
from pathlib import Path
import importlib
from stvirtual.models import stage1_3d as s1
from stvirtual.models import stage2_3d_transition as s2


In [ ]:
data_path = str(DATA_ROOT)
ckpt_path = str(CHECKPOINT_ROOT)
resl_path = str(RUN_ROOT)
lrpr_path = str(LR_PAIRS) if LR_PAIRS else None


In [ ]:
routes = CONFIG["routes"]
fracs = [f"{src}_to_{tgt}" for src, tgt in routes]
seg_key = fracs[0]
steps = 20


In [ ]:
if not SCANVI_ADATA.is_file():
    raise FileNotFoundError(f"Missing {SCANVI_ADATA}; run preprocess.ipynb first")
adata = sc.read_h5ad(SCANVI_ADATA)
for key in ["sample", "cell_type", "cx_aligned", "cy_aligned"]:
    if key not in adata.obs:
        raise KeyError(f"Preprocessing output is missing obs[{key!r}]")
if CONFIG["latent_key"] not in adata.obsm or "counts" not in adata.layers:
    raise KeyError("Preprocessing output must contain X_scanVI and raw counts")


In [ ]:
adata.obs['sample'].unique()


In [ ]:
model = None


## Train Stage 1

Learn transport between selected samples and save the Stage-1 checkpoint.


In [ ]:
res1 = {}
for route in routes:
    importlib.reload(s1)
    
    res1.update(s1.train_model_multislice(
        model=model,                  
        adata_all=adata,
        slice_key="sample",
        route_ids=route,
        save_root=str(STAGE1_CHECKPOINT_ROOT),
        x_key="cx_aligned",
        y_key="cy_aligned",
        latent_key=CONFIG["latent_key"],    
        steps=steps,  
        guide_eps=0.01,
        uot_eps=CONFIG["stage1_loss"]["uot_eps"], uot_tau=CONFIG["stage1_loss"]["uot_tau"], uot_lam_x=CONFIG["stage1_loss"]["uot_lam_x"], uot_lam_f=CONFIG["stage1_loss"]["uot_lam_f"],
        guide_topk=512, guide_temp=0.5, guide_schedule="linear",
        terrain_gap=False,
        cell_type_key='cell_type',
        lam_context=10.0,
        lam_residual=1.0,
        lam_vsmooth=0.1,  
        lam_uot=CONFIG["stage1_loss"]["lam_uot"],
        lib_layer='counts',              
        latent_dim=adata.obsm[CONFIG["latent_key"]].shape[1],             
        epochs=100,
        device="cuda:0" if torch.cuda.is_available() else "cpu",
    ))


In [ ]:
res1[fracs[0]].keys()


## Train the expression decoder

This step uses the preprocessed AnnData and the Stage-1 normalization from this run. Running decoder.ipynb separately is optional.


In [ ]:
sys.path.insert(0, str(EXPERIMENT_DIR))
from workflow import train_route_decoders, write_training_manifest

decoder_checkpoints = train_route_decoders(
    CONFIG, EXPERIMENT_DIR,
    device="cuda:0" if torch.cuda.is_available() else "cpu",
)
decoder_checkpoints


## Save Stage-1 traces and generate 2D boundaries

In [ ]:
from stvirtual.utils import boundary_2d as boundary
from stvirtual.utils.stage1_results import save_res as save_stage1_results
from stvirtual.utils.trajectory import simulation_trace_from_out

boundary_cfg = CONFIG["boundary"]
saved = save_stage1_results(
    res1=res1, adata_all=adata,
    out_dir=str(CHECKPOINT_ROOT / "stage1_res"),
    slice_key="sample", ann_key='cell_type',
    save_prefix="simulation_stage1", steps=steps,
    n_cache=boundary_cfg["n_cache"], unnormalize=False,
)

for stage_key in fracs:
    source_name = stage_key.split("_to_", 1)[0]
    match = re.search(r"\d+", source_name)
    source_number = int(match.group(0)) if match else 10**9
    trace = simulation_trace_from_out(res1[stage_key], steps=steps, n_cache=boundary_cfg["n_cache"], unnormalize=False)
    frames = [value.detach().cpu().numpy().astype(np.float32) for value in trace["x"]]
    stage_dir = RUN_ROOT / "bound" / stage_key
    stage_dir.mkdir(parents=True, exist_ok=True)
    statistics = []
    for frame_index, coordinates in enumerate(frames):
        shell, fraction, outside, expansion = boundary.make_shell_adaptive(
            coordinates, alpha_factor=boundary_cfg["alpha_factor"],
            seed=2026 + source_number + frame_index,
            n_resample=boundary_cfg["n_resample"], target_frac=boundary_cfg["target_frac"],
            expand0=boundary_cfg["expand0"], expand_step=boundary_cfg["expand_step"],
            expand_max=boundary_cfg["expand_max"], fallback_expand=boundary_cfg["fallback_expand"],
        )
        boundary.save_shell_csv(shell, stage_dir / f"bound_z{frame_index:03d}.csv")
        boundary.plot_shell_coverage(coordinates, shell, outside, stage_dir / f"bound_z{frame_index:03d}.png")
        statistics.append({"frame": frame_index, "n_cells": len(coordinates), "fraction_inside": fraction, "n_outside": len(outside), "expansion": expansion})
    pd.DataFrame(statistics).to_csv(stage_dir / "bound_stats.csv", index=False)

print("Stage-1 traces:", saved)
print("Boundary root:", RUN_ROOT / "bound")


## Train Stage 2

Load Stage-1 traces and boundaries, then train and save the Stage-2 model.


In [ ]:
importlib.reload(s2)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

ctx = s2.build_global_ctx(
    adata_path=str(SCANVI_ADATA),
    lr_pairs_path=lrpr_path,
    ckpt_3dslice=str(STAGE1_CHECKPOINT_ROOT / fracs[0] / "checkpoints" / "best.pt"),
    device=device,
    layer_col='cell_type',
)

stages = []
for sk in fracs:
    src, tgt = sk.split("_to_", 1)
    stages.append(
        s2.StageCfg(
            src=src,
            tgt=tgt,
            out_npz_path=saved[sk],
            bound_dir=f"{resl_path}/bound/{sk}",
            decoder_checkpoint=str(experiment_path(CONFIG["decoder_checkpoint"].format(src=src, tgt=tgt))),
            scanvi_dir=None,
            model_type="scanvi",
            use_lr=True,
            lr_source="decoder",
            use_latent=True,
            latent_key=CONFIG["latent_key"],
            z_csv_offset=0,
            layer_col='cell_type',
            diff_csv=str(DIFF_MAP),
        )
    )

outs = s2.run_multi_stages(
    ctx=ctx,
    sample_key = 'sample',
    stages=stages,
    best_ckpt_dir=str(STAGE2_CHECKPOINT_ROOT),
    train_kwargs=dict(EPOCHS=100, LR=1e-4, ADVECT_LATENT=True),
    rl_xy=CONFIG["stage2_loss"]["rl_xy"],
    rl_z=CONFIG["stage2_loss"]["rl_z"],
)

for o in outs:
    print(o["best_ckpt_path"])   


In [ ]:
ckpt_map = {}
for o in outs:
    cfg = o["stage"]
    ckpt_map[f"{cfg.src}_to_{cfg.tgt}"] = o["best_ckpt_path"]

ckpt_map


## Simulate and save results

Save simulation frames to the `output_dir` set below.


In [ ]:
importlib.reload(s2)
simulations = {}
for cfg in stages:
    seg_key = f"{cfg.src}_to_{cfg.tgt}"
    simulations[seg_key] = s2.simulation_policy_one_stage(
        s2, ctx, cfg, ckpt_map[seg_key],
        seed=2026, ADVECT_LATENT=True,
        output_dir=Path(resl_path) / 'simulation' / seg_key, output_prefix=seg_key,
    )
    print(seg_key, "Tp1=", len(simulations[seg_key]["coords"]))


## Save transition identity and differentiation metadata


In [ ]:
SIMULATION_OBS_FIELDS = ("uid", "parent_uid", "is_diff", "diff_alpha", "diff_tgt_layer", "diff_enter_step", "commit_step")

def persist_simulation_obs(simulation_result):
    output_paths = simulation_result.get("output_paths")
    if not isinstance(output_paths, list):
        raise KeyError("simulation_result has no output_paths; pass output_dir to simulation_policy_one_stage")
    if len(output_paths) != len(simulation_result["coords"]):
        raise ValueError("output_paths and simulation frames have different lengths")

    for frame_index, output_path in enumerate(output_paths):
        frame_adata = sc.read_h5ad(output_path)
        for field in SIMULATION_OBS_FIELDS:
            frames = simulation_result.get(field)
            if not isinstance(frames, list):
                raise KeyError(f"simulation_result does not provide {field!r}")
            values = np.asarray(frames[frame_index])
            if values.ndim != 1 or len(values) != frame_adata.n_obs:
                raise ValueError(
                    f"{field} at frame {frame_index} has shape {values.shape}; "
                    f"expected ({frame_adata.n_obs},)"
                )
            frame_adata.obs[field] = values
        frame_adata.obs_names = frame_adata.obs["uid"].astype(str).to_numpy()
        frame_adata.write_h5ad(output_path, compression="gzip")

    return {"frames_updated": len(output_paths), "obs_fields": list(SIMULATION_OBS_FIELDS)}

simulation_obs_reports = {
    route: persist_simulation_obs(route_simulation)
    for route, route_simulation in simulations.items()
}
simulation_obs_reports


In [ ]:

ro = simulations[seg_key]

for t in range(len(ro["coords"])):
    N  = int(np.asarray(ro["coords"][t]).shape[0])
    nb = int(np.asarray(ro["is_birth"][t]).sum())  
    print(f"{seg_key} | t={t:02d} | N={N} | new_birth={nb}")


## Connect this training run to the epicardial-cell perturbation workflow


In [ ]:
for stage in stages:
    route = f"{stage.src}_to_{stage.tgt}"
    manifest_path = experiment_path(CONFIG["training_manifest"].format(src=stage.src, tgt=stage.tgt))
    write_training_manifest(
        manifest_path, source=stage.src, target=stage.tgt, steps=steps,
        latent_key=CONFIG["latent_key"], adata_path=str(SCANVI_ADATA),
        stage1_trace=saved[route], bound_dir=stage.bound_dir,
        stage1_checkpoint=STAGE1_CHECKPOINT_ROOT / route / "checkpoints/best.pt",
        policy_checkpoint=ckpt_map[route], decoder_checkpoint=decoder_checkpoints[route],
        lr_pairs=LR_PAIRS, transition_prior=DIFF_MAP,
    )
    print("Extrapolation inputs:", manifest_path)
